# Snapshot Inicial: Ecosistema Politico en X (Twitter)

Recoleccion y analisis de tweets desde la inscripcion de candidaturas (13 de marzo 2026).

**Pasos:**
1. Resolver cuentas semilla (handles -> IDs)
2. Recolectar timelines de cuentas semilla
3. Buscar menciones de candidatos principales
4. Analizar hashtags y handles descubiertos

In [7]:
import sys
sys.path.insert(0, "..")

import polars as pl
import yaml
import json

from src.config import DATA_RAW, DATA_PROCESSED, PROJECT_ROOT
from src.twitter_client import get_client, TWEET_FIELDS, EXPANSIONS, USER_FIELDS
from src.collectors.tweet_collector import (
    SINCE_DATE, collect_user_timeline, resolve_user_ids,
    save_tweets, search_tweets,
)

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(80)

print(f"Recolectando desde: {SINCE_DATE}")

Recolectando desde: 2026-03-13T00:00:00Z


## 1. Cargar cuentas semilla y resolver IDs

Las cuentas estan en `config/candidates.yaml`, agrupadas por bloque (derecha/izquierda).

In [3]:
# Cargar configuracion y extraer todas las cuentas con IDs existentes
config_path = PROJECT_ROOT / "config" / "candidates.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# Aplanar todas las cuentas (candidatos + asociadas) de todos los bloques
all_accounts = [
    account
    for bloc in config["bloques"].values()
    for key in ("candidatos", "cuentas_asociadas")
    for account in bloc.get(key, [])
]

# id_map: handle_sin_@ -> user_id (solo los ya guardados)
id_map = {
    a["handle"].lstrip("@"): a["id"]
    for a in all_accounts
    if a.get("id", "").strip()
}

print(f"Cuentas totales: {len(all_accounts)}, IDs en YAML: {len(id_map)}")
print(id_map)

Cuentas totales: 10, IDs en YAML: 3
{'PalomaValenciaL': '149281495', 'ABDELAESPRIELLA': '1657059045956517897', 'IvanCepedaCast': '98781946'}


In [4]:
# Resolver handles sin ID y guardar en YAML
handles_missing = [a["handle"] for a in all_accounts if a["handle"].lstrip("@") not in id_map]
print(f"Sin ID: {handles_missing}")

if handles_missing:
    new_ids = resolve_user_ids(handles_missing)  # retorna {username_sin_@: id}
    id_map.update(new_ids)

    for a in all_accounts:
        username = a["handle"].lstrip("@")
        if username in new_ids:
            a["id"] = new_ids[username]

    with open(config_path, "w") as f:
        yaml.dump(config, f, allow_unicode=True, default_flow_style=False, sort_keys=False)
    print(f"YAML actualizado: {new_ids}")

print(f"\nid_map completo ({len(id_map)}):")
for u, uid in sorted(id_map.items()):
    print(f"  @{u} -> {uid}")

Sin ID: ['@JDOviedoAr', '@CeDemocratico', '@AlvaroUribeVel', '@petrogustavo', '@PizarroMariaJo', '@GustavoBolivar', '@PactoHistorico']
YAML actualizado: {'JDOviedoAr': '219434063', 'CeDemocratico': '1115440213', 'AlvaroUribeVel': '61097151', 'petrogustavo': '49849732', 'PizarroMariaJo': '1362851972', 'GustavoBolivar': '50981729', 'PactoHistorico': '998190549353017345'}

id_map completo (10):
  @ABDELAESPRIELLA -> 1657059045956517897
  @AlvaroUribeVel -> 61097151
  @CeDemocratico -> 1115440213
  @GustavoBolivar -> 50981729
  @IvanCepedaCast -> 98781946
  @JDOviedoAr -> 219434063
  @PactoHistorico -> 998190549353017345
  @PalomaValenciaL -> 149281495
  @PizarroMariaJo -> 1362851972
  @petrogustavo -> 49849732


In [5]:
with open("../data/raw/tweets_bloc2.json", "w", encoding="utf-8") as f:
    json.dump(bloc_records, f, ensure_ascii=False, indent=4)

NameError: name 'json' is not defined

### 2a. Guardar bloque derecha desde memoria (sin re-consultar API)

`bloc_records` tiene los tweets de derecha que fallaron al guardar por tipos mixtos en IDs.
Normalizamos los IDs a string y guardamos directo.

In [ ]:
# Bloque derecha: cargar desde parquet si ya existe, si no convertir desde JSON
_parquet_derecha = DATA_RAW / "seed_timelines_derecha.parquet"

if _parquet_derecha.exists():
    df_derecha = pl.read_parquet(_parquet_derecha)
    bloc_records = df_derecha.to_dicts()
    print(f"Cargado desde disco: {len(df_derecha)} tweets ({_parquet_derecha.name})")
else:
    import json as _json
    from datetime import datetime
    with open(DATA_RAW / "tweets_bloc2.json", encoding="utf-8") as f:
        bloc_records = _json.load(f)
    df_derecha = pl.DataFrame(bloc_records, infer_schema_length=None)
    df_derecha = df_derecha.with_columns(pl.lit(datetime.now().isoformat()).alias("collected_at"))
    df_derecha = df_derecha.unique(subset=["tweet_id"])
    _parquet_derecha.parent.mkdir(parents=True, exist_ok=True)
    df_derecha.write_parquet(_parquet_derecha)
    print(f"Convertido y guardado: {len(df_derecha)} tweets")

seed_tweets = {"derecha": bloc_records}

### 2b. Recolectar solo bloque izquierda

Derecha ya esta guardado. Solo consultamos las cuentas de izquierda.

In [ ]:
# Bloque izquierda: cargar desde parquet si ya existe, si no recolectar del API
_parquet_izq = DATA_RAW / "seed_timelines_izquierda.parquet"

if _parquet_izq.exists():
    df_izq = pl.read_parquet(_parquet_izq)
    izq_records = df_izq.to_dicts()
    print(f"Cargado desde disco: {len(df_izq)} tweets ({_parquet_izq.name})")
    seed_tweets["izquierda"] = izq_records
else:
    import json as _json
    from datetime import datetime
    izq_records = []
    for handle in blocs["izquierda"]:
        username = handle.lstrip("@")
        user_id = id_map.get(username)
        if not user_id:
            print(f"  No se pudo resolver {handle}, saltando")
            continue
        print(f"Recolectando @{username}... ", end="")
        records = collect_user_timeline(user_id)
        print(f"{len(records)} tweets")
        izq_records.extend(records)

    # Respaldo JSON
    with open(DATA_RAW / "tweets_izquierda.json", "w", encoding="utf-8") as f:
        _json.dump(izq_records, f, ensure_ascii=False, indent=4)

    df_izq = pl.DataFrame(izq_records, infer_schema_length=None)
    df_izq = df_izq.with_columns(pl.lit(datetime.now().isoformat()).alias("collected_at"))
    df_izq = df_izq.unique(subset=["tweet_id"])
    df_izq.write_parquet(_parquet_izq)
    print(f"Guardado: {len(df_izq)} tweets en {_parquet_izq.name}")
    seed_tweets["izquierda"] = izq_records

total = sum(len(v) for v in seed_tweets.values())
print(f"Total semillas: {total} tweets en {len(seed_tweets)} bloques")

In [ ]:
# (celda consolidada en la anterior)

## 3. Red de interaccion desde tweets existentes

En vez de consultar la lista de seguidos de cada semilla (costo: $0.01 por cuenta seguida),
construimos la red de interaccion a partir de los tweets ya recolectados.

Extraemos tres tipos de conexion:
- **Mentions**: a quien mencionan las semillas en sus tweets
- **Retweets/Quotes**: de quien amplifican contenido
- **Replies**: con quien conversan

Esto produce una red de interaccion real (mas util para THANOS que los follows estaticos)
y tiene **costo cero** porque usa datos ya descargados.

In [ ]:
# Cargar todos los tweets recolectados
parquet_files = list(DATA_RAW.glob("*.parquet"))
df_all = pl.concat([pl.read_parquet(f) for f in parquet_files])
df_all = df_all.unique(subset=["tweet_id"])
print(f"Tweets totales: {len(df_all)}")

seed_handles = {h.lower() for h in id_map.keys()}

# --- 3.1 Edges por mentions ---
edges_mentions = (
    df_all
    .select("author_username", "mentions")
    .explode("mentions")
    .drop_nulls("mentions")
    .rename({"author_username": "source", "mentions": "target"})
    .with_columns(
        pl.col("source").str.to_lowercase(),
        pl.col("target").str.to_lowercase(),
        pl.lit("mention").alias("edge_type"),
    )
)

# --- 3.2 Edges por retweets/quotes (ref_type -> author del tweet original) ---
#     Solo podemos vincular al autor original si esta en nuestros datos
ref_tweets = (
    df_all
    .filter(pl.col("ref_type").is_in(["retweeted", "quoted"]))
    .select("author_username", "ref_tweet_id", "ref_type")
)
# Mapear ref_tweet_id -> autor original
tweet_author_map = df_all.select(
    pl.col("tweet_id").alias("ref_tweet_id"),
    pl.col("author_username").alias("original_author"),
)
edges_retweets = (
    ref_tweets
    .join(tweet_author_map, on="ref_tweet_id", how="inner")
    .select(
        pl.col("author_username").str.to_lowercase().alias("source"),
        pl.col("original_author").str.to_lowercase().alias("target"),
        pl.col("ref_type").alias("edge_type"),
    )
)

# --- 3.3 Edges por replies ---
# in_reply_to_user_id -> necesitamos mapear a username
user_id_to_handle = df_all.select(
    pl.col("author_id").alias("in_reply_to_user_id"),
    pl.col("author_username").str.to_lowercase().alias("target"),
).unique(subset=["in_reply_to_user_id"])

edges_replies = (
    df_all
    .filter(pl.col("in_reply_to_user_id").is_not_null())
    .select("author_username", "in_reply_to_user_id")
    .join(user_id_to_handle, on="in_reply_to_user_id", how="inner")
    .select(
        pl.col("author_username").str.to_lowercase().alias("source"),
        pl.col("target"),
        pl.lit("reply").alias("edge_type"),
    )
)

# --- Combinar todas las aristas ---
edges = pl.concat([edges_mentions, edges_retweets, edges_replies])
# Eliminar self-loops
edges = edges.filter(pl.col("source") != pl.col("target"))

print(f"Aristas totales: {len(edges)}")
print(f"  Mentions: {len(edges_mentions)}")
print(f"  Retweets/Quotes: {len(edges_retweets)}")
print(f"  Replies: {len(edges_replies)}")
print(f"  Nodos unicos: {edges.select(pl.concat_list(['source', 'target']).alias('n')).explode('n').n_unique()}")

In [ ]:
import plotly.express as px

# --- 3.4 Nodos mas conectados (in-degree: cuantas veces son mencionados/RT/replied) ---
in_degree = (
    edges
    .group_by("target")
    .agg(
        pl.len().alias("in_degree"),
        pl.col("edge_type").filter(pl.col("edge_type") == "mention").len().alias("mentions_in"),
        pl.col("edge_type").filter(pl.col("edge_type").is_in(["retweeted", "quoted"])).len().alias("rt_in"),
        pl.col("edge_type").filter(pl.col("edge_type") == "reply").len().alias("replies_in"),
        pl.col("source").n_unique().alias("unique_sources"),
    )
    .sort("in_degree", descending=True)
)

# Marcar si es cuenta semilla
in_degree = in_degree.with_columns(
    pl.col("target").is_in(list(seed_handles)).alias("is_seed")
)

print("Top 30 cuentas mas referenciadas en la red:")
print(in_degree.head(30))

# --- 3.5 Cuentas NO semilla mas relevantes (candidatas a expandir red) ---
non_seed_top = in_degree.filter(~pl.col("is_seed")).head(30)
print("\nTop 30 cuentas NO semilla mas referenciadas (red organica):")
print(non_seed_top)

# --- 3.6 Visualizacion: top 20 por in-degree ---
plot_data = in_degree.head(20).to_pandas()
fig = px.bar(
    plot_data,
    x="target",
    y=["mentions_in", "rt_in", "replies_in"],
    title="Top 20 cuentas por interacciones recibidas (desglose por tipo)",
    labels={"target": "Cuenta", "value": "Interacciones", "variable": "Tipo"},
    barmode="stack",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# --- 3.7 Out-degree: quien interactua mas (detectar bots o cuentas de campana) ---
out_degree = (
    edges
    .group_by("source")
    .agg(
        pl.len().alias("out_degree"),
        pl.col("target").n_unique().alias("unique_targets"),
    )
    .sort("out_degree", descending=True)
)
print("\nTop 20 cuentas que mas interactuan (posibles bots/campana):")
print(out_degree.head(20))

# --- 3.8 Guardar red ---
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
edges.write_parquet(DATA_PROCESSED / "interaction_edges.parquet")
in_degree.write_parquet(DATA_PROCESSED / "interaction_in_degree.parquet")
print("\nGuardado: interaction_edges.parquet, interaction_in_degree.parquet")

## 4. Analisis de los datos recolectados

Cargamos todos los parquets recolectados y analizamos hashtags, handles y autores.

In [ ]:
# Cargar todos los parquets recolectados
parquet_files = list(DATA_RAW.glob("*.parquet"))
print(f"Archivos encontrados: {len(parquet_files)}")
for f in parquet_files:
    df_tmp = pl.read_parquet(f)
    print(f"  {f.name}: {len(df_tmp)} tweets")

df = pl.concat([pl.read_parquet(f) for f in parquet_files])
df = df.unique(subset=["tweet_id"])
print(f"\nTotal tweets unicos: {len(df)}")

### 4.1 Top hashtags

Estos son los hashtags mas usados en el ecosistema politico recolectado.
Los top 10 por bloque son los que alimentan la variable `x_{t,h,l}^(k)` del modelo THANOS.

In [ ]:
hashtags_df = (
    df.select("tweet_id", "hashtags")
    .explode("hashtags")
    .drop_nulls("hashtags")
    .group_by(pl.col("hashtags").str.to_lowercase().alias("hashtag"))
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(f"Hashtags unicos encontrados: {len(hashtags_df)}\n")
print("Top 30 hashtags:")
hashtags_df.head(30)

### 4.2 Top handles mencionados

Cuentas mas mencionadas en los tweets recolectados. Candidatos a ser agregados al tracking.

In [ ]:
mentions_df = (
    df.select("tweet_id", "mentions")
    .explode("mentions")
    .drop_nulls("mentions")
    .group_by(pl.col("mentions").str.to_lowercase().alias("handle"))
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(f"Handles unicos mencionados: {len(mentions_df)}\n")
print("Top 30 handles mencionados:")
mentions_df.head(30)

### 4.3 Autores mas activos

Usuarios que mas publican sobre los candidatos. Utiles para identificar influenciadores
y tambien posibles bots (cuentas con actividad anormalmente alta).

In [ ]:
authors_df = (
    df.group_by("author_username")
    .agg(
        pl.len().alias("tweet_count"),
        pl.col("retweet_count").sum().alias("total_retweets"),
        pl.col("like_count").sum().alias("total_likes"),
        pl.col("author_followers").first().alias("followers"),
        pl.col("author_created_at").first().alias("account_created"),
    )
    .sort("tweet_count", descending=True)
)

print("Top 30 autores mas activos:")
authors_df.head(30)

### 4.4 Volumen de tweets por dia

Visualizacion del volumen de actividad diaria desde la inscripcion de candidaturas.

In [ ]:
import plotly.express as px

daily = (
    df.with_columns(pl.col("created_at").str.slice(0, 10).alias("date"))
    .group_by("date")
    .agg(pl.len().alias("tweets"))
    .sort("date")
)

fig = px.bar(
    daily.to_pandas(),
    x="date", y="tweets",
    title="Volumen diario de tweets recolectados",
    labels={"date": "Fecha", "tweets": "Tweets"},
)
fig.show()

### 4.5 Distribucion de tipos de tweet

Proporcion de tweets originales, retweets, replies y quotes.

In [ ]:
type_counts = (
    df.with_columns(
        pl.col("ref_type").fill_null("original").alias("tweet_type")
    )
    .group_by("tweet_type")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

fig = px.pie(
    type_counts.to_pandas(),
    values="count", names="tweet_type",
    title="Distribucion por tipo de tweet",
)
fig.show()

## 4b. Expansion de red: desde las interacciones detectadas

Usamos las cuentas NO semilla mas referenciadas (seccion 3) como candidatas a expansion.
Buscamos tweets de esas cuentas usando hashtags de campana (busqueda, no timeline).

**Estimacion de costo antes de ejecutar.**

In [ ]:
from src.cost_estimator import estimate_search, estimate_user_lookup, print_estimate

# Cuentas no-semilla mas relevantes de la red de interaccion
expansion_candidates = non_seed_top.head(20).get_column("target").to_list()
print(f"Candidatas a expansion ({len(expansion_candidates)}): {expansion_candidates[:10]}...")

# Top hashtags (minimo 5 apariciones)
TOP_N_HASHTAGS = 20
MIN_HASHTAG_COUNT = 5
top_hashtags = (
    hashtags_df
    .filter(pl.col("count") >= MIN_HASHTAG_COUNT)
    .head(TOP_N_HASHTAGS)
    .get_column("hashtag")
    .to_list()
)
print(f"Hashtags de campana ({len(top_hashtags)}): {top_hashtags}")

# --- Estimar costo ANTES de ejecutar ---
# Una busqueda por grupo de hashtags, max 5 paginas = ~500 tweets
est_search = estimate_search(query_count=1, tweets_per_query=500)
# Lookup de las cuentas candidatas para obtener metricas publicas
est_users = estimate_user_lookup(len(expansion_candidates))

BUDGET = 10.0  # USD - ajustar segun presupuesto disponible
total = print_estimate(est_search, est_users, budget=BUDGET)

if total > BUDGET:
    print("\n** Costo excede presupuesto. Ajustar parametros antes de continuar. **")

In [ ]:
from importlib import reload
import src.collectors.tweet_collector as tc
reload(tc)

# Buscar tweets con hashtags de campana
hashtag_query = " OR ".join(f"#{h}" for h in top_hashtags)
query = f"({hashtag_query[:400]}) lang:es -is:retweet"
print(f"Query ({len(query)} chars): {query[:200]}...")

search_tweets = tc.search_tweets(query, max_pages=5, use_full_archive=True)

# Filtrar por cuentas de la red (semillas + candidatas de expansion)
allowed = seed_handles | {c.lower() for c in expansion_candidates}
filtered = [t for t in search_tweets if (t.get("author_username") or "").lower() in allowed]

print(f"\nTweets encontrados: {len(search_tweets)}")
print(f"De cuentas en la red: {len(filtered)}")

tc.save_tweets(filtered, "expanded_network_tweets")

## 5. Guardar rankings

Almacena los rankings calculados en `data/processed/` para uso posterior en el modelo.

In [ ]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

hashtags_df.write_parquet(DATA_PROCESSED / "hashtag_rankings.parquet")
mentions_df.write_parquet(DATA_PROCESSED / "mention_rankings.parquet")
authors_df.write_parquet(DATA_PROCESSED / "author_rankings.parquet")

print("Rankings guardados en data/processed/:")
print(f"  hashtag_rankings.parquet  ({len(hashtags_df)} hashtags)")
print(f"  mention_rankings.parquet  ({len(mentions_df)} handles)")
print(f"  author_rankings.parquet   ({len(authors_df)} autores)")

## 6. Exploracion libre

Celda para consultas ad-hoc sobre los datos recolectados.

In [ ]:
# Ejemplo: tweets con mas engagement
df.select(
    "author_username", "text", "retweet_count", "like_count", "created_at"
).sort("like_count", descending=True).head(20)

In [ ]:
# Ejemplo: buscar un hashtag especifico
# hashtag_a_buscar = "colombia"
# df.filter(
#     pl.col("hashtags").list.contains(hashtag_a_buscar)
# ).select("author_username", "text", "created_at").head(10)